# Hermite Moment Density Estimator

This notebook demonstrates the complete-only pipeline in `moment_density_estimator_complete.py`.



## Complete Objective

Exact regime (`extra_hermite_terms=0`): minimize the coefficient-space density-L2 anchor `(c - c_ref)^T G (c - c_ref)` under nonnegativity and optional unit-mass constraints.

Extended regime (`extra_hermite_terms>0`): minimize `alpha * (A c - mu_*)^T W (A c - mu_*) + lambda_eff * (c - c_ref)^T G (c - c_ref)`, where `lambda_eff = moment_qp_complete_lambda * moment_qp_complete_coeff_prior_mult`.


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
sys.path.insert(0, str(_root))

import numpy as np
import scipy.stats as st
from IPython.display import display

from moment_density_estimator_complete import (
    BimodalNormal,
    empirical_moments_from_samples,
    hermite_complete_chain_report,
)


## Run a Complete-Only Report

The only QP metric in this notebook is the complete objective. The controls to tune first are `extra_hermite_terms`, `moment_qp_complete_alpha`, `moment_qp_complete_lambda`, and `moment_qp_complete_coeff_prior_mult`.


In [ ]:
np.random.seed(41)

# Choose one of: "bimodal", "normal", "logistic", "skewed", "su".
_example = "logistic"

_GUIDING_EXAMPLES = {
    "bimodal": (
        BimodalNormal(mu1=-2.0, sigma1=1.0, mu2=2.0, sigma2=1.0, w=0.5),
        np.linspace(1.2, 2.9, 50),
    ),
    "normal": (
        st.norm(loc=0.0, scale=1.0),
        np.linspace(1.4, 2.0, 50),
    ),
    "logistic": (
        st.logistic(loc=0.0, scale=np.sqrt(3.0) / np.pi),
        np.linspace(1.9, 2.7, 50),
    ),
    "skewed": (
        st.johnsonsu(1.08, 2.18, loc=1.0, scale=1.76),
        np.linspace(2.0, 3.2, 50),
    ),
    "su": (
        st.johnsonsu(0.0, 1.8, loc=0.0, scale=1.6),
        np.linspace(2.0, 3.2, 50),
    ),
}

_dist, _a_grid = _GUIDING_EXAMPLES[_example]
_n_samples = 50_000
_max_moment_order = 40
_prefix_len = 8

_samples = _dist.rvs(_n_samples)
_moments_full = empirical_moments_from_samples(_samples, _max_moment_order)
_moments_prefix = _moments_full[:_prefix_len].copy()

_x = np.linspace(-8.0, 8.0, 129)

_res = hermite_complete_chain_report(
    _moments_prefix,
    _dist,
    _n_samples,
    x=_x,
    a_grid=_a_grid,
    extra_hermite_terms=3,
    moment_qp_complete_alpha=1.0,
    moment_qp_complete_lambda=1e-6,
    moment_qp_complete_coeff_prior_mult=1.0,
    moment_space_ridge_G=1e-9,
    gram_n_grid=1001,
    plot_metric_curves=True,
    plot_summary_figure=True,
)

sorted(_res.keys())


## Table and Figures

The table has one row for each extension level `k=0,...,extra_hermite_terms`.


In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(_res["table_rows"]))
except Exception:
    print(_res["table_rows"])

for _fig_key in ("figure", "figure_metrics", "figure_density_and_mise", "figure_summary"):
    if _res.get(_fig_key) is not None:
        display(_res[_fig_key])


## Command-Line Equivalent

The same workflow can be run from a terminal by choosing one of the five guiding examples:

```bash
python run_moment_density_estimator_complete.py --demo --dist logistic --output-dir reports/logistic --extra-hermite-terms 3 --alpha 1.0 --lambda 1e-6 --coeff-prior-mult 1.0
```

Available choices are `bimodal`, `normal`, `logistic`, `skewed`, and `su`. Each choice uses its built-in default bandwidth grid unless you override it with `--a-min`, `--a-max`, and `--a-points`.

For real data, pass `--samples path/to/data.csv` or `--moments path/to/moments.csv`. If the true distribution is unknown, omit `--dist`; if it matches one of the five examples, include `--dist` to show the true PDF.
